In [4]:
import os
import xarray as xr
import geopandas as gpd
from shapely.geometry import Point
import rioxarray as rxr

data_dir = "/storage/dlhogan/precipitation-rodeo/data/"

# Sand Castle 1: Merging SAIL datasets

In [2]:
original_file_dir = "/storage/dlhogan/precipitation-rodeo/data/raw/SAIL"
filename = "SAIL_met_all.nc"

In [ ]:
ds = xr.open_dataset(os.path.join(original_file_dir, filename))

In [ ]:
# save the merged dataset to a new NetCDF file
lat = ds["lat"].values[0]
lon = ds["lon"].values[0]
elev = ds["alt"].values[0]

# Create a GeoDataFrame and name after dataset.attrs['datastream']

gdf = gpd.GeoDataFrame(
    {
        "datastream": [ds.attrs.get("datastream", "unknown")],
        "latitude": [lat],
        "longitude": [lon],
        "elevation": [elev],
    },
    geometry=[Point(lon, lat, elev)],
    crs="EPSG:4326",
)

# Sand Castle 2: Extracting PRISM data for specific basin

In [2]:
def clip_prism(raster_path, shape_path):
    # Prepare output path
    out_path = raster_path.replace("raw", "processed").replace(".nc", "_clipped.nc")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Open raster
    with xr.open_dataset(raster_path) as ds:
        # Read shapefile
        shape = gpd.read_file(shape_path)

        # Set CRS for raster
        crs = ds.crs.attrs["crs_wkt"]
        ds = ds.rio.write_crs(crs)

        # Transform shapefile to match raster CRS
        shape = shape.to_crs(ds.rio.crs)

        # Clip raster
        clipped = ds.rio.clip(shape.geometry, shape.crs, drop=True)

        # Rename variable to ppt and set attributes
        clipped = clipped.rename({"Band1": "ppt"})
        clipped["ppt"].attrs["units"] = "mm"
        clipped["ppt"].attrs["long_name"] = "PRISM daily precipitation"

        # Write clipped file
        clipped.to_netcdf(out_path)

    # Delete original only if the clipped file was successfully written
    if os.path.exists(out_path):
        os.remove(raster_path)
        print(f"✅ Original deleted: {raster_path}")

    return

In [3]:
example_path = "/storage/dlhogan/precipitation-rodeo/data/external/PRISM/raw/prism_ppt_us_30s_20201023.nc"
shape_path = "/storage/dlhogan/precipitation-rodeo/data/geographic/East_River_lumped_HRUs_GRUs.shp"

clip_prism(example_path, shape_path)

✅ Original deleted: /storage/dlhogan/precipitation-rodeo/data/external/PRISM/raw/prism_ppt_us_30s_20201023.nc


In [3]:
ds = xr.open_dataset("/storage/dlhogan/precipitation-rodeo/data/external/PRISM/processed/prism_ppt_us_30s_20201024_clipped.nc")

In [4]:
ds

<xarray.Dataset> Size: 8kB
Dimensions:  (lat: 44, lon: 44)
Coordinates:
  * lat      (lat) float64 352B 38.67 38.68 38.68 38.69 ... 39.01 39.02 39.03
  * lon      (lon) float64 352B -107.1 -107.1 -107.1 ... -106.8 -106.8 -106.8
Data variables:
    crs      int64 8B ...
    ppt      (lat, lon) float32 8kB ...
Attributes:
    Conventions:  CF-1.5
    GDAL:         GDAL 3.4.3, released 2022/04/22
    history:      Tue Oct 14 15:33:15 2025: GDAL CreateCopy( /nfs/pancake/u5/...

# Sand Castle 3: ERA5-Land Data

In [ ]:
import cdsapi
import pandas as pd
import xarray as xr

# Set the time zone shift as a variable so it is easy to change
TIME_ZONE_SHIFT_HOURS = -7  # UTC-7 for MDT

# Calculate the time the TIME-ZONE midnight in UTC
LOCAL_MIDNIGHT_IN_UTC = (0-TIME_ZONE_SHIFT_HOURS) % 24
TIME_STEPS = ['00:00', f"{LOCAL_MIDNIGHT_IN_UTC:02d}:00"]
client = cdsapi.Client() 
dataset = "reanalysis-era5-land"
request = {
    'product_type': ['reanalysis'],
    'variable': ['total_precipitation'],
    'date': '20240101/20240131',
    'time': TIME_STEPS,
    'area': [39.1, -107.1, 38.8, -106.8],  # North, West, South, East
    'grid': [1, 1],
    'data_format': 'netcdf',
}
result_file = client.retrieve(dataset, request).download("/storage/dlhogan/precipitation-rodeo/data/external/ERA5-Land/era5_land_20240101_20240131.zip")

2025-10-14 16:29:41,700 INFO Request ID is c4ec8237-bfd1-41c3-8c56-ed474c6ef812
2025-10-14 16:29:41,913 INFO status has been updated to accepted
2025-10-14 16:29:56,158 INFO status has been updated to successful


6e1e79d8ae6dc38756a3bf3f37402ce1.zip:   0%|          | 0.00/25.2k [00:00<?, ?B/s]

In [28]:
result_file

'/storage/dlhogan/precipitation-rodeo/data/external/ERA5-Land/era5_land_20240101_20240131.zip'

In [30]:
import zipfile
import os

# unzip if needed
if result_file.endswith(".zip"):
    with zipfile.ZipFile(result_file, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(result_file))
    # Remove the zip file after extraction
    os.remove(result_file)
    result_file = result_file.replace(".zip", ".nc")
    # rename the file to era5_land_20240101_20240131.nc
    new_file_path = os.path.join(os.path.dirname(result_file), "era5_land_20240101_20240131.nc")
    os.rename(result_file, new_file_path)
    result_file = new_file_path
ds = xr.open_dataset(
    "/storage/dlhogan/precipitation-rodeo/data/external/ERA5-Land/era5_land_20240101_20240131.nc"
)
ds

<xarray.Dataset> Size: 2kB
Dimensions:     (valid_time: 62, latitude: 1, longitude: 1)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 496B 2024-01-01 ... 2024-01-31T07...
  * latitude    (latitude) float64 8B 38.8
  * longitude   (longitude) float64 8B -107.1
    expver      (valid_time) <U4 992B ...
Data variables:
    tp          (valid_time, latitude, longitude) float32 248B ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-14T23:19 GRIB to CDM+CF via cfgrib-0.9.1...

In [14]:
# Group the data by hour
ds_grouped_by_hour = ds.groupby("valid_time.hour")

# Then create new datasets for the UTC midnight and the local midnight
i_UTC_minight, i_local_midnight = ds_grouped_by_hour.groups
ds_UTC_midnight = ds.isel(valid_time=ds_grouped_by_hour.groups[i_UTC_minight])
ds_local_midnight = ds.isel(valid_time=ds_grouped_by_hour.groups[i_local_midnight])

ds_local_midnight = ds_local_midnight.assign_coords(
    valid_time=ds_UTC_midnight.valid_time + pd.Timedelta(days=1)
)

# Subtract the UTC midnight data from the local midnight data
ds_local_to_utc_midnight = ds_UTC_midnight - ds_local_midnight
# Shift the time back one day
ds_local_to_utc_midnight = ds_local_to_utc_midnight.assign_coords(
    valid_time=ds_local_to_utc_midnight.valid_time - pd.Timedelta(days=1)
)

ds_accum_local = ds_local_midnight + ds_local_to_utc_midnight

shift = int(TIME_ZONE_SHIFT_HOURS < 0)
ds_accum_local = ds_accum_local.assign_coords(
    valid_time=ds_accum_local.valid_time + pd.Timedelta(days=shift)
)

# Sand Castle 4: Testing laser disdrometer output

In [39]:
ds = xr.open_dataset(f"{data_dir}/processed/SPLASH/SPLASH_kp_laser_disdrometer_30min.nc")

# Sand Castle 5: Working on processing AOS met data

In [5]:
import glob
import os
os.chdir("/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/")
from utils import process_sail_data
from utils.helper_funcs import convert_to_local_time
import numpy as np
from scipy import stats

files = glob.glob(f"{data_dir}/raw/SAIL/aos_mtcb/*.nc")
example_ds = xr.open_dataset(files[1])

In [6]:
precipitation_sum_vars = [v for v in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] if v in example_ds.data_vars]
precipitation_duration_vars = [v for v in process_sail_data.SAIL_PRECIPITATION_VARS['duration'] if v in example_ds.data_vars]
precipitation_rate_vars = [v for v in process_sail_data.SAIL_PRECIPITATION_VARS['rate'] if v in example_ds.data_vars]
relative_humidity_vars = [v for v in process_sail_data.SAIL_HUMIDITY_VARS['mean'] if v in example_ds.data_vars]
wind_spd_vars = [v for v in process_sail_data.SAIL_WIND_VARS['mean'] if v in example_ds.data_vars] + ["u", "v"]
wind_dir_vars = [v for v in process_sail_data.SAIL_WIND_VARS['median'] if v in example_ds.data_vars]
temperature_vars = [v for v in process_sail_data.SAIL_TEMPERATURE_VARS['mean'] if v in example_ds.data_vars]
pressure_vars = [v for v in process_sail_data.SAIL_PRESSURE_VARS['mean'] if v in example_ds.data_vars]

KeyError: 'median'

In [7]:
# calculate u and v from wind speed and direction
def calculate_wind_components(wind_speed, wind_direction):
    """
    Calculate the u and v components of wind from wind speed and direction.

    Parameters:
    wind_speed (float or np.ndarray): Wind speed in m/s.
    wind_direction (float or np.ndarray): Wind direction in degrees from north.

    Returns:
    tuple: A tuple containing the u and v components of the wind.
    """
    # Convert wind direction from degrees to radians
    wind_direction_rad = np.radians(wind_direction)

    # Calculate u and v components
    u = -wind_speed * np.sin(wind_direction_rad)  # East-West component
    v = -wind_speed * np.cos(wind_direction_rad)  # North-South component

    return u.data, v.data
def xr_mode(x, axis=None):
    """Compute the statistical mode for an xarray reduce operation."""
    mode_result = stats.mode(x, nan_policy='omit', axis=axis)
    return xr.DataArray(mode_result.mode)


u, v = calculate_wind_components(example_ds['wind_speed'].values, example_ds['wind_direction'].values)

# Add u and v to the dataset
example_ds['u'] = (('time'), u)
example_ds['v'] = (('time'), v)
example_ds['u'].attrs['units'] = 'm/s'
example_ds['v'].attrs['units'] = 'm/s'
example_ds['u'].attrs['long_name'] = 'East-West wind component'
example_ds['v'].attrs['long_name'] = 'North-South wind component'

example_ds = convert_to_local_time(example_ds, local_tz='America/Denver')

In [71]:
precipitation_sum_da      = example_ds[precipitation_sum_vars].resample(time='30min').sum()
precipitation_duration_da = example_ds[precipitation_duration_vars].resample(time='30min').sum()
precipitation_rate_da     = example_ds[precipitation_rate_vars].resample(time='30min').mean()
relative_humidity_da      = example_ds[relative_humidity_vars].resample(time='30min').mean()
wind_spd_da               = example_ds[wind_spd_vars].resample(time='30min').mean()
wind_dir_da               = example_ds[wind_dir_vars].resample(time='30min').reduce(xr_mode)
temperature_da            = example_ds[temperature_vars].resample(time='30min').mean()
pressure_da               = example_ds[pressure_vars].resample(time='30min').mean()

In [73]:
# assign units to the new data arrays
for var in precipitation_sum_vars:
    precipitation_sum_da[var].attrs['units'] = 'mm'
for var in precipitation_duration_vars:
    # convert from seconds to minutes if needed
    if precipitation_duration_da[var].attrs['units'] == 's':
        precipitation_duration_da[var] = precipitation_duration_da[var] / 60
        precipitation_duration_da[var].attrs['units'] = 'min'
for var in precipitation_rate_vars:
    precipitation_rate_da[var].attrs['units'] = 'mm/hr'
for var in relative_humidity_vars:
    relative_humidity_da[var].attrs['units'] = '%'
for var in wind_spd_vars:
    wind_spd_da[var].attrs['units'] = 'm/s'
for var in wind_dir_vars:
    wind_dir_da[var].attrs['units'] = 'deg'
for var in temperature_vars:
    temperature_da[var].attrs['units'] = 'degC'
for var in pressure_vars:
    pressure_da[var].attrs['units'] = 'hPa'

In [74]:
ds_merged = xr.merge([
        precipitation_sum_da,
        precipitation_duration_da,
        precipitation_rate_da,
        relative_humidity_da,
        wind_spd_da,
        wind_dir_da,
        temperature_da,
        pressure_da
    ])

# Sand Castle 6

In [ ]:
import glob
import os
import sys, os
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
os.chdir(project_root)
from utils import process_sail_data
from utils.helper_funcs import convert_to_local_time
import xarray as xr
import time
import numpy as np
from scipy import stats
import pandas as pd

# Assign data directory and get files
SITE_NAME = "gothic"
data_dir = "/storage/dlhogan/precipitation-rodeo/data/"
files = glob.glob(f"{data_dir}raw/SAIL/laser_disdrometer_{SITE_NAME}/*.nc")

In [25]:
parsivel_correction_dict = { 
    'holroyd1971': [0.17, -1],
    'brandes2007': [0.178, -0.922],
    'heymsfield2004': [0.104, -0.95]
}

def correct_SAIL_parsivel_for_snow(ds, method='holroyd1971'):
    """
    Correct snowfall rate using a method discussed in Boudala et al. 2014
    """
    a = parsivel_correction_dict[method][0]
    b = parsivel_correction_dict[method][1]
    # Number density of particles
    N_D = ds['number_density_drops']
    # Fall velocity of particles summed over raw_fall_velocity
    V_D = ds['fall_velocity_calculated']
    # Class size width
    class_size_width = ds['class_size_width']

    # Apply the condition to include particle sizes from 2 to 31
    particle_size_indices = range(2, 32)
    raw_fall_velocity_indices = range(2, 32)

    # Select the relevant slices using isel
    N_D_masked = N_D.isel(particle_size=particle_size_indices)
    class_size_width_masked = class_size_width.isel(particle_size=particle_size_indices)
    V_D_masked = V_D.isel(raw_fall_velocity=raw_fall_velocity_indices)

    # Calculate the snowfall rate using vectorized operations
    result = (N_D_masked * V_D_masked * class_size_width_masked ** (3 + b)).sum(dim='particle_size').sum(dim='raw_fall_velocity')

    # Calculate the final result
    final_result = (6 * a * np.pi * 10e-4 * result)/60
    # filter to only include times with snowfall
    final_result = final_result.where(ds['weather_code'].isin([70,71,72,73,74,75,76,77,78,79,85,86,87]), ds['precip_rate'])
    return final_result

1. read data file (1-minute resolution)
2. filter to only variables we want to keep
3. filter out any bad data
4. calculate correction for snow using all methods (save these as individual arrays)
5. create accumulated variable
6. convert time to local time
7. resample to desired length using appropriate function for each variable
8. 

In [ ]:
ds = xr.open_dataset(files[0])
vars_to_keep = [
    'precip_rate', # mean
    'weather_code', # mode
    'equivalent_radar_reflectivity_ott', # mean or max
    'number_detected_particles', # sum
    'mor_visibility', # mode
    'class_size_width', # mean
    'fall_velocity_calculated', # mean
    'liquid_water_content', # mean
    'raw_spectrum', # mean?
    'median_volume_diameter', # median
    'snow_depth_intensity', # mean
    'number_density_drops', # mean
    'lon',
    'lat',
    'alt',
    ]

for var in vars_to_keep:
    if 'qc_' + var in ds.data_vars:
        vars_to_keep.append('qc_' + var)

ds_sub = ds[vars_to_keep]

In [22]:
for var in ds_sub.data_vars:
    if 'qc' in var:
        data_var = var.replace('qc_', '')
        # drop replace values with NaN where qc is not 0
        ds_sub[data_var] = ds_sub[data_var].where(ds_sub[var] == 0)

In [26]:
corrected_ds = correct_SAIL_parsivel_for_snow(ds_sub, 'holroyd1971')